In [80]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [81]:
from sklearn import set_config
set_config(display='text')

In [82]:
normal = pd.read_csv(r'C:\DL_proj\normal_dataset.csv')
conf   = pd.read_csv(r'C:\DL_proj\confidential_dataset.csv')

print(normal.shape, conf.shape)
print(normal.columns.tolist())
normal.head(2)

(470, 3) (640, 3)
['filename', 'text', 'label']


,filename,text,label
0,edge_bhy_174.txt,[특허 관련 문서] 등록 특허 목록 안내\n작성부서: 법무팀 (내선 3815)\n사...,0
1,edge_jsb_172.txt,[대외 협의] 사무용품 공급업체 입찰 공고 안내\n작성부서: 구매기획팀 (내선 36...,0


In [83]:
df = pd.concat([normal, conf], ignore_index=True)

# 라벨 제대로 들어갔는지 확인
print(normal['label'].unique())   # [0] 이어야 함
print(conf['label'].unique())     # [1] 이어야 함

print(df.shape)
print(df['label'].value_counts())
print(df['text'].duplicated().sum())
print(df.isna().sum())
print(df['text'].str.len().describe())

[0]
[1]
(1110, 3)
label
1    640
0    470
Name: count, dtype: int64
0
filename    0
text        0
label       0
dtype: int64
count    1110.000000
mean      505.163063
std       161.306572
min       197.000000
25%       367.250000
50%       480.500000
75%       594.000000
max       916.000000
Name: text, dtype: float64


In [71]:
df['prefix'] = df['filename'].str.split('_').str[0]
print(pd.crosstab(df['prefix'], df['label']))

label     0    1
prefix          
design    0  150
edge     20    0
hr        0  150
normal  450    0
pii       0  170
sales     0  170


In [72]:
df.isna().sum()
df['text'].duplicated().sum()
df['label'].value_counts()
df['text'].str.len().describe()

count    1110.000000
mean      505.163063
std       161.306572
min       197.000000
25%       367.250000
50%       480.500000
75%       594.000000
max       916.000000
Name: text, dtype: float64

In [73]:
print(df[df['prefix']=='edge']['label'].value_counts())

label
0    20
Name: count, dtype: int64


In [74]:
X = df['text']
y = df['label']

In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y, 
    test_size= 0.2,
    random_state=42,
    stratify=y
)
print(len(X_train),len(X_test))

888 222


테스트 1) TF-IDF + LR

In [94]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# --- 로드 ---
normal = pd.read_csv(r'C:\DL_proj\normal_dataset.csv')
conf   = pd.read_csv(r'C:\DL_proj\confidential_dataset.csv')
df = pd.concat([normal, conf], ignore_index=True)

# --- split ---
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# --- 벡터화 + 학습 ---
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1,2),
                        min_df=2, max_features=20000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_vec, y_train)

# --- 평가 ---
pred = clf.predict(X_test_vec)
print(classification_report(y_test, pred, target_names=['NORMAL','CONFIDENTIAL'], digits=3))
print(confusion_matrix(y_test, pred))

# --- 계수 ---
names, co = tfidf.get_feature_names_out(), clf.coef_[0]
print('--- CONFIDENTIAL ---')
for i in np.argsort(co)[::-1][:15]: print(f'{names[i]:18s} {co[i]:+.2f}')
print('--- NORMAL ---')
for i in np.argsort(co)[:15]:       print(f'{names[i]:18s} {co[i]:+.2f}')

              precision    recall  f1-score   support

      NORMAL      1.000     1.000     1.000        94
CONFIDENTIAL      1.000     1.000     1.000       128

    accuracy                          1.000       222
   macro avg      1.000     1.000     1.000       222
weighted avg      1.000     1.000     1.000       222

[[ 94   0]
 [  0 128]]
--- CONFIDENTIAL ---
2025년              +1.35
한빛반도체              +1.09
연봉                 +1.09
2026               +0.95
담당자                +0.93
채용                 +0.89
출입                 +0.88
2024년              +0.84
2026년              +0.81
계약                 +0.79
비상                 +0.77
내부                 +0.76
제목                 +0.76
공급                 +0.75
한빛반도체 내부           +0.74
--- NORMAL ---
사내                 -1.72
내선                 -1.33
동호회                -1.33
업무                 -1.26
주십시오               -1.22
교육                 -1.18
공용                 -1.07
주시기                -1.01
안내                 -1.01
주시기 바랍니다      

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

normal = pd.read_csv(r'C:\DL_proj\normal_dataset.csv')
conf   = pd.read_csv(r'C:\DL_proj\confidential_dataset.csv')
df = pd.concat([normal, conf], ignore_index=True)

def normalize(t):
    t = re.sub(r'20\d{2}\s*년?', ' ', t)
    t = re.sub(r'\d+\s*[월일]', ' ', t)
    t = re.sub(r'[\$￦]?\s?\d[\d,]*\s*(원|만원|억원|달러)', ' ', t)
    t = re.sub(r'\d+\s*%p?', ' ', t)
    t = re.sub(r'\d[\d,.]*', ' ', t)
    return re.sub(r'\s+', ' ', t)

df['norm'] = df['text'].apply(normalize)    # 모든 행에 normalize함수 적용

STOP = ['제목','담당자','작성자','작성부서','배포대상',
        '발신','수신','문서','문서번호','내선']

In [91]:
X, y = df['norm'], df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1,2),
                        min_df=2, max_features=20000, stop_words=STOP)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [92]:
pred = clf.predict(X_test_vec)
print(classification_report(y_test, pred, target_names=['NORMAL','CONFIDENTIAL'], digits=3))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

      NORMAL      1.000     1.000     1.000        94
CONFIDENTIAL      1.000     1.000     1.000       128

    accuracy                          1.000       222
   macro avg      1.000     1.000     1.000       222
weighted avg      1.000     1.000     1.000       222

[[ 94   0]
 [  0 128]]


In [93]:
names, co = tfidf.get_feature_names_out(), clf.coef_[0]
print('--- CONFIDENTIAL ---')
for i in np.argsort(co)[::-1][:15]: print(f'{names[i]:18s} {co[i]:+.2f}')
print('--- NORMAL ---')
for i in np.argsort(co)[:15]:       print(f'{names[i]:18s} {co[i]:+.2f}')

--- CONFIDENTIAL ---
연봉                 +1.22
한빛반도체              +1.21
채용                 +1.02
내부                 +0.91
출입                 +0.91
한빛반도체 내부           +0.83
공정코드 hb            +0.82
담당부서               +0.82
공정코드               +0.82
인상률                +0.82
계약                 +0.80
비상                 +0.80
hb                 +0.79
검토                 +0.78
내부 hb              +0.75
--- NORMAL ---
사내                 -1.88
동호회                -1.41
주십시오               -1.37
업무                 -1.28
교육                 -1.24
주시기                -1.07
안내                 -1.07
공용                 -1.06
주시기 바랍니다           -1.05
공지                 -1.05
문의                 -0.93
예약                 -0.91
모집                 -0.90
사내 공지              -0.89
필요한                -0.83


In [95]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_train_vec, y_train)
print(classification_report(y_test, rf.predict(X_test_vec), digits=3))

              precision    recall  f1-score   support

           0      1.000     1.000     1.000        94
           1      1.000     1.000     1.000       128

    accuracy                          1.000       222
   macro avg      1.000     1.000     1.000       222
weighted avg      1.000     1.000     1.000       222

